In [1]:
import numpy as np
import pandas as pd

In [2]:
class NaiveBayesClassifier:
    def fit(self, trainingTable, correctAnswers):
        trainingTable = np.asarray(trainingTable, dtype=np.float64)
        correctAnswers = np.asarray(correctAnswers)

        self.categories = np.unique(correctAnswers)

        numberOfTrainingTypes = trainingTable.shape[1]
        numberOfCategories = len(self.categories)

        self.averageTrainingValues = np.zeros((numberOfCategories, numberOfTrainingTypes), dtype=np.float64)
        self.trainingSpreads = np.zeros((numberOfCategories, numberOfTrainingTypes), dtype=np.float64)
        self.categoryBaseProbabilities = np.zeros(numberOfCategories, dtype=np.float64)

        for categoryIndex, categoryName in enumerate(self.categories):
            trainingForThisCategory = trainingTable[correctAnswers == categoryName]

            self.averageTrainingValues[categoryIndex, :] = trainingForThisCategory.mean(axis=0)
            self.trainingSpreads[categoryIndex, :] = trainingForThisCategory.var(axis=0) + 1e-9

            totalRows = trainingTable.shape[0]
            categoryRowCount = trainingForThisCategory.shape[0]
            self.categoryBaseProbabilities[categoryIndex] = categoryRowCount / float(totalRows)

    def _calculateTrainingProbability(self, categoryIndex, singleItemTraining):
        avg = self.averageTrainingValues[categoryIndex]
        spread = self.trainingSpreads[categoryIndex]

        numerator = np.exp(-((singleItemTraining - avg) ** 2) / (2 * spread))
        denominator = np.sqrt(2 * np.pi * spread)
        return numerator / denominator

    def _predictSingleItem(self, singleItemTraining):
        categoryScores = []

        for categoryIndex in range(len(self.categories)):
            baseScore = np.log(self.categoryBaseProbabilities[categoryIndex])

            trainingProbabilities = self._calculateTrainingProbability(categoryIndex, singleItemTraining)
            combinedTrainingScore = np.sum(np.log(trainingProbabilities))

            totalScore = baseScore + combinedTrainingScore
            categoryScores.append(totalScore)

        winningIndex = np.argmax(categoryScores)
        return self.categories[winningIndex]

    def predict(self, testingTable):
        testingTable = np.asarray(testingTable, dtype=np.float64)
        return np.array([self._predictSingleItem(itemTraining) for itemTraining in testingTable])


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def runKaggleDataset(csvFilePath, targetColumnName):
    print(f"--- Processing: {csvFilePath.split("/")[-1].split(".")[0]} ---")

    df = pd.read_csv(csvFilePath)

    for idCol in ['id', 'Id', 'ID', 'Unnamed: 0']:
        if idCol in df.columns:
            df = df.drop(columns=[idCol])

    df = df.fillna(df.mean(numeric_only=True))

    rawFeatures = df.drop(columns=[targetColumnName])
    correctAnswers = df[targetColumnName]

    trainingTable = pd.get_dummies(rawFeatures, drop_first=True)

    trainingData, testingData, answersTrain, answersTest = train_test_split(
        trainingTable, correctAnswers, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier()
    classifier.fit(trainingData, answersTrain)
    predictions = classifier.predict(testingData)

    accuracy = accuracy_score(answersTest, predictions) * 100
    print(f"Accuracy: {accuracy:.2f}%\n")

In [5]:
import kagglehub
heartPath = kagglehub.dataset_download("johnsmith88/heart-disease-dataset") + "/heart.csv"
breastCancerPath = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data") + "/data.csv"
diabetesPath = kagglehub.dataset_download("jamaltariqcheema/pima-indians-diabetes-dataset") + "/diabetes.csv"
irisPath = kagglehub.dataset_download("uciml/iris") + "/Iris.csv"
waterPath = kagglehub.dataset_download("adityakadiwal/water-potability") + "/water_potability.csv"

100%|██████████| 6.18k/6.18k [00:00<00:00, 8.92MB/s]

Extracting files...


100%|██████████| 48.6k/48.6k [00:00<00:00, 194kB/s]

Extracting files...


100%|██████████| 9.13k/9.13k [00:00<00:00, 8.36MB/s]

Extracting files...


100%|██████████| 3.60k/3.60k [00:00<00:00, 6.35MB/s]

Extracting files...


100%|██████████| 251k/251k [00:00<00:00, 259kB/s]

Extracting files...


In [6]:
runKaggleDataset(heartPath, "target")

--- Processing: heart ---
Accuracy: 80.00%



In [7]:
runKaggleDataset(breastCancerPath, "diagnosis")

--- Processing: data ---
Accuracy: 62.28%



In [8]:
runKaggleDataset(diabetesPath, "Outcome")

--- Processing: diabetes ---
Accuracy: 76.62%



In [9]:
runKaggleDataset(irisPath, "Species")

--- Processing: Iris ---
Accuracy: 100.00%



In [10]:
runKaggleDataset(waterPath, "Potability")

--- Processing: water_potability ---
Accuracy: 63.11%

